# Multilingual Brand-Safety Comparison

Load the consolidated LLM ratings and inspect where brand-safety or suitability opinions diverge across language versions of the same article.

In [ ]:
from pathlib import Path

import pandas as pd
# data file is in the repository root
DATA_PATH = Path("euronews_llm_ratings_consolidated_apr2.csv")
df = pd.read_csv(DATA_PATH)
df

In [ ]:
# ANOVA for between-run and between-language differences in level ratings only
# Stable categories/titles with >=2 languages; title fixed effects are absorbed for speed.
import pandas as pd
import numpy as np
from scipy import sparse
from scipy.stats import f as f_dist

stable_df = df.copy()
if 'category_unstable' in stable_df.columns:
    stable_df = stable_df.loc[~stable_df['category_unstable'].fillna(True)].copy()

title_lang_counts = stable_df.groupby('rss_title')['language'].nunique()
multi_lang_titles = title_lang_counts[title_lang_counts >= 2].index
anova_df = stable_df.loc[stable_df['rss_title'].isin(multi_lang_titles), [
    'rss_title', 'language', 'run_index', 'category', 'level'
]].copy()

level_map = {'0': 0, 'low': 1, 'medium': 2, 'high': 3}
anova_df['level_score'] = anova_df['level'].astype(str).str.lower().map(level_map)
anova_df = anova_df.dropna(subset=['language', 'run_index', 'category', 'rss_title', 'level_score']).copy()
anova_df['run_label'] = 'run' + anova_df['run_index'].astype(int).astype(str)
level_anova_df = anova_df.copy()

# Mean level scores by run and language (rows=runs, cols=languages).
level_table = level_anova_df.pivot_table(
    index='run_label', columns='language', values='level_score', aggfunc='mean'
)
print('Mean level score (0-3) by run and language')
display(level_table.round(3))


def _treatment_dummies(codes, n_levels, n_rows):
    """Sparse treatment-coded dummies, dropping the first level as reference."""
    if n_levels <= 1:
        return sparse.csr_matrix((n_rows, 0), dtype=float)
    mask = codes > 0
    rows = np.flatnonzero(mask)
    cols = codes[mask] - 1
    return sparse.csr_matrix(
        (np.ones(rows.size, dtype=float), (rows, cols)),
        shape=(n_rows, n_levels - 1),
        dtype=float,
    )


def _interaction_dummies(left_codes, right_codes, n_left, n_right, n_rows):
    if n_left <= 1 or n_right <= 1:
        return sparse.csr_matrix((n_rows, 0), dtype=float)
    mask = (left_codes > 0) & (right_codes > 0)
    rows = np.flatnonzero(mask)
    cols = (left_codes[mask] - 1) * (n_right - 1) + (right_codes[mask] - 1)
    return sparse.csr_matrix(
        (np.ones(rows.size, dtype=float), (rows, cols)),
        shape=(n_rows, (n_left - 1) * (n_right - 1)),
        dtype=float,
    )


def _fit_from_crossprod(XtX, Xty, yty, cols):
    cols = np.asarray(cols, dtype=int)
    if cols.size == 0:
        return float(yty), 0
    A = XtX[np.ix_(cols, cols)]
    b = Xty[cols]
    A = 0.5 * (A + A.T)
    eigvals, eigvecs = np.linalg.eigh(A)
    keep = eigvals > 1e-8
    rank = int(keep.sum())
    if rank == 0:
        return float(yty), 0
    projected = np.einsum('ij,i->j', eigvecs[:, keep], b, optimize=True)
    model_sum_sq = float(np.sum((projected**2) / eigvals[keep]))
    rss = float(yty) - model_sum_sq
    return max(rss, 0.0), rank


def _crossprod_with_intercept(X, y):
    XtX = X.T.dot(X).toarray()
    Xty = np.asarray(X.T.dot(y)).ravel()
    yty = float(np.einsum('i,i->', y, y, optimize=True))
    col_sums = np.asarray(X.sum(axis=0)).ravel()

    XtX_intercept = np.empty((X.shape[1] + 1, X.shape[1] + 1), dtype=float)
    XtX_intercept[0, 0] = X.shape[0]
    XtX_intercept[0, 1:] = col_sums
    XtX_intercept[1:, 0] = col_sums
    XtX_intercept[1:, 1:] = XtX
    Xty_intercept = np.r_[y.sum(), Xty]
    return XtX_intercept, Xty_intercept, yty, XtX, Xty


def _add_f_row(rows, name, rss_fit, rank_fit, rss_reduced, rank_reduced, mse_full, df_resid):
    df_num = rank_fit - rank_reduced
    sum_sq = rss_reduced - rss_fit
    if sum_sq < 0 and abs(sum_sq) < 1e-7:
        sum_sq = 0.0
    if df_num > 0:
        f_value = (sum_sq / df_num) / mse_full
        p_value = f_dist.sf(f_value, df_num, df_resid)
    else:
        f_value = np.nan
        p_value = np.nan
    rows.append({'term': name, 'sum_sq': sum_sq, 'df': df_num, 'F': f_value, 'PR(>F)': p_value})


def _two_way_level_anova(level_df):
    work = level_df[['run_label', 'language', 'level_score']].copy()
    for col in ['run_label', 'language']:
        work[col] = pd.Categorical(work[col], categories=sorted(work[col].dropna().unique()))

    y = work['level_score'].to_numpy(dtype=float)
    n = len(work)
    run_codes = work['run_label'].cat.codes.to_numpy(dtype=np.int64)
    lang_codes = work['language'].cat.codes.to_numpy(dtype=np.int64)
    n_run = work['run_label'].cat.categories.size
    n_lang = work['language'].cat.categories.size

    X_run = _treatment_dummies(run_codes, n_run, n)
    X_lang = _treatment_dummies(lang_codes, n_lang, n)
    X_inter = _interaction_dummies(run_codes, lang_codes, n_run, n_lang, n)

    blocks = [X_run, X_lang, X_inter]
    names = ['C(run_label)', 'C(language)', 'C(run_label):C(language)']
    term_cols = {}
    start = 0
    for name, block in zip(names, blocks):
        stop = start + block.shape[1]
        term_cols[name] = np.arange(start, stop)
        start = stop

    X = sparse.hstack(blocks, format='csr')
    XtX_i, Xty_i, yty, _, _ = _crossprod_with_intercept(X, y)
    raw_cols = np.arange(X.shape[1])

    def with_intercept(cols):
        return np.r_[0, np.asarray(cols, dtype=int) + 1]

    def fit(cols):
        return _fit_from_crossprod(XtX_i, Xty_i, yty, with_intercept(cols))

    rss_full, rank_full = fit(raw_cols)
    df_resid = n - rank_full
    mse_full = rss_full / df_resid
    no_inter_cols = np.setdiff1d(raw_cols, term_cols['C(run_label):C(language)'], assume_unique=True)
    rss_no_inter, rank_no_inter = fit(no_inter_cols)

    rows = []
    _add_f_row(
        rows,
        'C(run_label)',
        rss_no_inter,
        rank_no_inter,
        *fit(np.setdiff1d(no_inter_cols, term_cols['C(run_label)'], assume_unique=True)),
        mse_full,
        df_resid,
    )
    _add_f_row(
        rows,
        'C(language)',
        rss_no_inter,
        rank_no_inter,
        *fit(np.setdiff1d(no_inter_cols, term_cols['C(language)'], assume_unique=True)),
        mse_full,
        df_resid,
    )
    _add_f_row(
        rows,
        'C(run_label):C(language)',
        rss_full,
        rank_full,
        rss_no_inter,
        rank_no_inter,
        mse_full,
        df_resid,
    )
    rows.append({'term': 'Residual', 'sum_sq': rss_full, 'df': df_resid, 'F': np.nan, 'PR(>F)': np.nan})
    return pd.DataFrame(rows).set_index('term')


def _absorbed_title_type2_anova(level_df):
    work = level_df[['rss_title', 'run_label', 'language', 'category', 'level_score']].copy()
    for col in ['rss_title', 'run_label', 'language', 'category']:
        work[col] = pd.Categorical(work[col], categories=sorted(work[col].dropna().unique()))

    y = work['level_score'].to_numpy(dtype=float)
    n = len(work)
    run_codes = work['run_label'].cat.codes.to_numpy(dtype=np.int64)
    lang_codes = work['language'].cat.codes.to_numpy(dtype=np.int64)
    cat_codes = work['category'].cat.codes.to_numpy(dtype=np.int64)
    title_codes = work['rss_title'].cat.codes.to_numpy(dtype=np.int64)

    n_run = work['run_label'].cat.categories.size
    n_lang = work['language'].cat.categories.size
    n_cat = work['category'].cat.categories.size
    n_title = work['rss_title'].cat.categories.size

    X_run = _treatment_dummies(run_codes, n_run, n)
    X_lang = _treatment_dummies(lang_codes, n_lang, n)
    X_cat = _treatment_dummies(cat_codes, n_cat, n)
    X_inter = _interaction_dummies(lang_codes, cat_codes, n_lang, n_cat, n)

    blocks = [X_run, X_lang, X_cat, X_inter]
    names = ['C(run_label)', 'C(language)', 'C(category)', 'C(language):C(category)']
    term_cols = {}
    start = 0
    for name, block in zip(names, blocks):
        stop = start + block.shape[1]
        term_cols[name] = np.arange(start, stop)
        start = stop

    X = sparse.hstack(blocks, format='csr')
    all_cols = np.arange(X.shape[1])

    # Raw cross-products for the small, non-title part of the design.
    XtX = X.T.dot(X).toarray()
    Xty = np.asarray(X.T.dot(y)).ravel()
    yty = float(np.einsum('i,i->', y, y, optimize=True))

    # Frisch-Waugh-Lovell absorption of rss_title fixed effects:
    # residualize y and every non-title design column by title using cross-products.
    title_indicator = sparse.csr_matrix(
        (np.ones(n, dtype=float), (np.arange(n), title_codes)),
        shape=(n, n_title),
        dtype=float,
    )
    title_counts = np.bincount(title_codes, minlength=n_title).astype(float)
    title_sum_X = title_indicator.T.dot(X).toarray()
    title_sum_y = np.bincount(title_codes, weights=y, minlength=n_title).astype(float)
    inv_counts = 1.0 / title_counts

    XtX_absorbed = XtX - np.einsum('gi,gj,g->ij', title_sum_X, title_sum_X, inv_counts, optimize=True)
    Xty_absorbed = Xty - np.einsum('gi,g,g->i', title_sum_X, title_sum_y, inv_counts, optimize=True)
    yty_absorbed = yty - float(np.sum(title_sum_y**2 * inv_counts))

    def fit_absorbed(cols):
        return _fit_from_crossprod(XtX_absorbed, Xty_absorbed, yty_absorbed, cols)

    rss_full, rank_x_full = fit_absorbed(all_cols)
    rank_full_total = n_title + rank_x_full
    df_resid = n - rank_full_total
    mse_full = rss_full / df_resid

    rows = []
    no_inter_cols = np.setdiff1d(all_cols, term_cols['C(language):C(category)'], assume_unique=True)
    rss_no_inter, rank_no_inter = fit_absorbed(no_inter_cols)

    # Type-II-style tests: main language/category effects are tested before the interaction.
    _add_f_row(
        rows,
        'C(run_label)',
        rss_full,
        rank_x_full,
        *fit_absorbed(np.setdiff1d(all_cols, term_cols['C(run_label)'], assume_unique=True)),
        mse_full,
        df_resid,
    )
    _add_f_row(
        rows,
        'C(language)',
        rss_no_inter,
        rank_no_inter,
        *fit_absorbed(np.setdiff1d(no_inter_cols, term_cols['C(language)'], assume_unique=True)),
        mse_full,
        df_resid,
    )
    _add_f_row(
        rows,
        'C(category)',
        rss_no_inter,
        rank_no_inter,
        *fit_absorbed(np.setdiff1d(no_inter_cols, term_cols['C(category)'], assume_unique=True)),
        mse_full,
        df_resid,
    )
    _add_f_row(
        rows,
        'C(language):C(category)',
        rss_full,
        rank_x_full,
        rss_no_inter,
        rank_no_inter,
        mse_full,
        df_resid,
    )

    # Test the absorbed title effects against the same non-title design plus an intercept.
    XtX_no_title, Xty_no_title, _, _, _ = _crossprod_with_intercept(X, y)
    rss_no_title, rank_no_title = _fit_from_crossprod(
        XtX_no_title, Xty_no_title, yty, np.arange(XtX_no_title.shape[0])
    )
    _add_f_row(
        rows,
        'C(rss_title)',
        rss_full,
        rank_full_total,
        rss_no_title,
        rank_no_title,
        mse_full,
        df_resid,
    )

    rows.append({'term': 'Residual', 'sum_sq': rss_full, 'df': df_resid, 'F': np.nan, 'PR(>F)': np.nan})
    anova = pd.DataFrame(rows).set_index('term')

    diagnostics = pd.DataFrame([{
        'n_obs': n,
        'n_titles_absorbed': n_title,
        'n_runs': n_run,
        'n_languages': n_lang,
        'n_categories': n_cat,
        'non_title_design_columns': X.shape[1],
        'absorbed_non_title_rank': rank_x_full,
        'residual_df': df_resid,
    }])
    return anova, diagnostics


level_anova = _two_way_level_anova(level_anova_df)
print('Two-way ANOVA for level scores (0-3)')
display(level_anova.round(3))

full_level_anova, absorbed_anova_diagnostics = _absorbed_title_type2_anova(level_anova_df)

display(absorbed_anova_diagnostics)
print('Full fixed-effects ANOVA for level scores: run + language + category + language:category + absorbed title')
display(full_level_anova.round(3))


In [ ]:
from pathlib import Path

out_dir = Path("notebook_outputs")
out_dir.mkdir(exist_ok=True)

level_anova.to_csv(out_dir / "level_anova.csv")
full_level_anova.to_csv(out_dir / "full_level_anova.csv")
absorbed_anova_diagnostics.to_csv(out_dir / "full_level_anova_diagnostics.csv", index=False)

level_anova.to_html(out_dir / "level_anova.html", float_format=lambda x: f"{x:.3f}")

full_level_anova.to_html(out_dir / "full_level_anova.html", float_format=lambda x: f"{x:.3f}")


In [ ]:
# Filter out category-level instability across runs (drop categories where runs disagree)
group_cols = ['rss_title', 'language', 'category']
floor_spread = df.groupby(group_cols)['floor'].transform(lambda s: s.dropna().astype(str).str.lower().nunique())
level_spread = df.groupby(group_cols)['level'].transform(lambda s: s.dropna().astype(str).str.lower().nunique())
df['category_unstable'] = (floor_spread > 1) | (level_spread > 1)
df = df.loc[~df['category_unstable'].fillna(True)].copy()
df


In [ ]:
# Stable share by category/language with percent and counts
pivot = (
    df[['rss_title', 'language', 'category']]
      .dropna(subset=['rss_title', 'language', 'category'])
      .drop_duplicates()
      .groupby(['category', 'language'])['rss_title']
      .nunique()
      .unstack(fill_value=0)
      .sort_index()
)

pivot_stable = (
    df.loc[~df['unstable'].fillna(False), ['rss_title', 'language', 'category']]
      .dropna(subset=['rss_title', 'language', 'category'])
      .drop_duplicates()
      .groupby(['category', 'language'])['rss_title']
      .nunique()
      .unstack(fill_value=0)
      .reindex_like(pivot)
      .fillna(0)
)

stable_counts = pivot_stable.astype(int)
total_counts = pivot.astype(int)

ratio_pct = (
    stable_counts
    .div(total_counts.replace(0, pd.NA))
    .fillna(0)
    .mul(100)
)

def fmt_cell(cat, lang):
    stable = stable_counts.at[cat, lang] if lang in stable_counts.columns else 0
    total = total_counts.at[cat, lang] if lang in total_counts.columns else 0
    pct = ratio_pct.at[cat, lang] if lang in ratio_pct.columns else 0
    return f"{pct:.2f}% ({stable}/{total})"

display_table = ratio_pct.apply(lambda row: [fmt_cell(row.name, col) for col in ratio_pct.columns], axis=1)
display_table = pd.DataFrame(display_table.tolist(), index=ratio_pct.index, columns=ratio_pct.columns)
display_table

In [ ]:
# Pairwise language stats: category mismatch plus floor/level differences (stable rows, titles with >=2 languages)
from itertools import combinations

score_floor = {'safe': 0, 'unsafe': 1}
score_level = {'0': 0, 'low': 1, 'medium': 2, 'high': 3}

stable = (
    df.loc[~df['category_unstable'].fillna(False), ['rss_title', 'language', 'category', 'floor', 'level']]
      .dropna(subset=['rss_title', 'language', 'category', 'floor', 'level'])
)
stable = stable[stable['floor'].isin(score_floor) & stable['level'].isin(score_level)]

# keep only titles with 2+ language versions
multi_titles = stable.groupby('rss_title')['language'].nunique()
stable = stable[stable['rss_title'].isin(multi_titles[multi_titles >= 2].index)]


In [ ]:

# Per-category language ranking: avg level shift vs others (with significance)
from itertools import combinations
import numpy as np
import pandas as pd
from scipy.stats import ttest_1samp, wilcoxon
from statsmodels.stats.multitest import multipletests
import seaborn as sns
import matplotlib.pyplot as plt

score_level = {'0': 0, 'low': 1, 'medium': 2, 'high': 3}

if 'stable' not in globals() or stable is None:
    raise ValueError("Run the pairwise stats cell to define `stable` first")

cat_lang_rows = []
shift_rows = []

for cat, cat_df in stable.groupby('category'):
    cat_stats = []
    for title, g in cat_df.groupby('rss_title'):
        langs = g['language'].unique()
        if len(langs) < 2:
            continue
        for la, lb in combinations(langs, 2):
            a_scores = g.loc[g['language'] == la, 'level'].map(score_level).values
            b_scores = g.loc[g['language'] == lb, 'level'].map(score_level).values
            for a in a_scores:
                for b in b_scores:
                    cat_stats.append({'lang_i': la, 'lang_j': lb, 'd': a - b})

    if not cat_stats:
        continue

    stats_df = pd.DataFrame(cat_stats)
    stats_df_pos = stats_df[stats_df['d'] > 0]
    if stats_df_pos.empty:
        continue

    # contributions mirror the original scoring (positive differences only)
    contrib = []
    for _, row in stats_df_pos.iterrows():
        d = row['d']
        contrib.append({'category': cat, 'language': row['lang_i'], 'shift': d})
        contrib.append({'category': cat, 'language': row['lang_j'], 'shift': -d})

    contrib_df = pd.DataFrame(contrib)
    if contrib_df.empty:
        continue

    shift_rows.append(contrib_df)

    lang_stats = (
        contrib_df.groupby('language')['shift']
        .agg(avg_shift='mean', weight='size')
        .reset_index()
    )

    for _, r in lang_stats.iterrows():
        cat_lang_rows.append({
            'category': cat,
            'language': r['language'],
            'avg_level_shift_vs_others': round(r['avg_shift'], 3),
            'weight': int(r['weight'])
        })

cat_lang_rank = pd.DataFrame(cat_lang_rows)
if not cat_lang_rank.empty:
    lang_matrix = (
        cat_lang_rank
        .pivot(index='category', columns='language', values='avg_level_shift_vs_others')
        .sort_index()
    )
else:
    lang_matrix = pd.DataFrame()

display(cat_lang_rank, lang_matrix)

# ----- Significance tests per cell (H0: shift_vs_others = 0) -----
if shift_rows:
    shift_df = pd.concat(shift_rows, ignore_index=True)
else:
    shift_df = pd.DataFrame(columns=['category', 'language', 'shift'])

sig_rows = []
for (cat, lang), g in shift_df.groupby(['category', 'language']):
    x = g['shift'].dropna().values
    n = len(x)
    mean_shift = np.mean(x) if n else np.nan
    sd_shift = np.std(x, ddof=1) if n >= 2 else np.nan

    # one-sample t-test
    if n >= 2 and np.isfinite(sd_shift) and sd_shift > 0:
        _, t_p = ttest_1samp(x, popmean=0, alternative='two-sided')
    else:
        t_p = np.nan

    # Wilcoxon signed rank
    try:
        if n >= 1 and np.any(x != 0):
            _, w_p = wilcoxon(x, alternative='two-sided', zero_method='wilcox')
        else:
            w_p = np.nan
    except Exception:
        w_p = np.nan

    sig_rows.append({
        'category': cat,
        'language': lang,
        'n_obs': n,
        'mean_shift': mean_shift,
        'sd_shift': sd_shift,
        't_p': t_p,
        'wilcoxon_p': w_p,
    })

lang_sig = pd.DataFrame(sig_rows)

# multiple testing BH
for p_col in ['t_p', 'wilcoxon_p']:
    lang_sig[p_col + '_fdr'] = np.nan
    lang_sig[p_col + '_sig'] = False
    mask = lang_sig[p_col].notna()
    if mask.sum() > 0:
        reject, p_adj, _, _ = multipletests(lang_sig.loc[mask, p_col], method='fdr_bh')
        lang_sig.loc[mask, p_col + '_fdr'] = p_adj
        lang_sig.loc[mask, p_col + '_sig'] = reject

lang_sig = lang_sig.sort_values(['category', 'mean_shift'], ascending=[True, False]).reset_index(drop=True)
display(lang_sig)

# ----- Build star/annot matrices aligned to lang_matrix -----
p_col_main = 'wilcoxon_p_fdr'
if not lang_matrix.empty:
    pval_matrix = (
        lang_sig
        .pivot(index='category', columns='language', values=p_col_main)
        .reindex(index=lang_matrix.index, columns=lang_matrix.columns)
    )
    n_matrix = (
        lang_sig
        .pivot(index='category', columns='language', values='n_obs')
        .reindex(index=lang_matrix.index, columns=lang_matrix.columns)
    )
else:
    pval_matrix = pd.DataFrame()
    n_matrix = pd.DataFrame()

def p_to_stars(p):
    if pd.isna(p):
        return ''
    elif p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return ''

if not pval_matrix.empty:
    stars_matrix = pval_matrix.applymap(p_to_stars)
else:
    stars_matrix = pd.DataFrame()

if not lang_matrix.empty:
    lang_matrix_annot = lang_matrix.copy().astype(object)
    for r in lang_matrix.index:
        for c in lang_matrix.columns:
            val = lang_matrix.loc[r, c]
            star = stars_matrix.loc[r, c] if not stars_matrix.empty else ''
            lang_matrix_annot.loc[r, c] = np.nan if pd.isna(val) else f"{val:.2f}\n{star}"
else:
    lang_matrix_annot = pd.DataFrame()

display(pval_matrix.round(4) if not pval_matrix.empty else pval_matrix)
display(stars_matrix)
display(lang_matrix_annot)

# Category-level cross-language disparity: row-wise spread in heatmap values
if not lang_matrix.empty:
    category_disparity_stats = pd.DataFrame({
        'sd_across_languages': lang_matrix.std(axis=1, skipna=True),
        'range_across_languages': lang_matrix.max(axis=1, skipna=True) - lang_matrix.min(axis=1, skipna=True),
        'mean_abs_shift': lang_matrix.abs().mean(axis=1, skipna=True),
        'n_languages': lang_matrix.notna().sum(axis=1),
    }).sort_values('sd_across_languages', ascending=False)
else:
    category_disparity_stats = pd.DataFrame()

display(
    category_disparity_stats.style
    .format({'sd_across_languages': '{:.3f}', 'range_across_languages': '{:.3f}', 'mean_abs_shift': '{:.3f}', 'n_languages': '{:.0f}'})
    .set_caption('Category-level cross-language disparity: row-wise SD of directional shifts')
)

# ----- Heatmap with significance stars -----
if not lang_matrix.empty:
    plt.figure(figsize=(9.5, 6), dpi=600)
    sns.heatmap(
        lang_matrix,
        annot=lang_matrix_annot,
        fmt='',
        cmap='coolwarm',
        center=0,
        vmin=-1.5,
        vmax=1.75,
        linewidths=0.5
    )
    # plt.title('Avg level disparity vs others (if any) (positive = harsher)')
    plt.xlabel('Language')
    plt.ylabel('Category')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    # plt.savefig('output/avg_disparity_if_any.png', dpi=600)
    plt.show()


In [ ]:
# save results for the plot
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)
lang_matrix_annot.to_csv(output_dir / 'avg_level_shift_vs_others_if_any_annot.csv')
pval_matrix.to_csv(output_dir / 'avg_level_shift_vs_others_if_any_pvalues.csv', float_format='%.6f')
lang_sig.to_csv(output_dir / 'avg_level_shift_vs_others_if_any_significance.csv', index=False)
category_disparity_stats.to_csv(output_dir / 'category_cross_language_disparity_sd.csv', float_format='%.6f')  

In [ ]:
# ----- Heatmap with significance stars -----
if not lang_matrix.empty:
    plt.figure(figsize=(11, 5.55), dpi=600)
    sns.heatmap(
        lang_matrix,
        annot=lang_matrix_annot,
        fmt='',
        cmap='coolwarm',
        center=0,
        vmin=-1.5,
        vmax=1.75,
        linewidths=0.5,
    )
    # plt.title('Avg level disparity vs others (if any) (positive = harsher)')
    plt.xlabel('Language')
    plt.ylabel('Category')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig('output/avg_disparity_if_any_gemini_1000.png', dpi=600)
    plt.show()


In [ ]:
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10
plt.rcParams['font.weight'] = 'medium'

CATEGORY_MAP = {
    'adult_sexual':           'Adult/Sexual',
    'alcohol':                'Alcohol',
    'arms_ammunition':        'Arms/Ammunition',
    'crime':                  'Crime',
    'death_injury_military':  'Death/Injury/Military',
    'hate_speech':            'Hate Speech',
    'illegal_drugs':          'Illegal Drugs',
    'misinformation':         'Misinformation',
    'obscenity_profanity':    'Obscenity/Profanity',
    'online_piracy':          'Online Piracy',
    'sensitive_social_issues':'Sensitive Social Issues',
    'spam_malware':           'Spam/Malware',
    'terrorism':              'Terrorism',
    'tobacco_vaping':         'Tobacco/Vaping',
}

# Apply mapping to your matrices
lang_matrix       = lang_matrix.rename(index=CATEGORY_MAP)
lang_matrix_annot = lang_matrix_annot.rename(index=CATEGORY_MAP)


if not lang_matrix.empty:
    plt.figure(figsize=(9.5, 6))
    ax=sns.heatmap(
        lang_matrix,
        annot=lang_matrix_annot,
        fmt='',
        cmap='coolwarm',
        center=0,
        vmin=-1.5,
        vmax=1.75,
        linewidths=0.5
    )
    # plt.title('Avg level disparity vs others (if any) (positive = harsher)')
    plt.xlabel('Language Code')
    plt.ylabel('Category')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig('avg_disparity_if_any.pdf', dpi=1200)
    plt.show()